# S6E6 — Second Phase (Project 기반)

이전 판과 달라진 점만 먼저:

| 예전 | 지금 |
|---|---|
| `PipelineBuilder(path='exp', name=...)` | `project.pipeline_builder(name)` |
| `p.set_grp(..., role='stage'/'head')` | `role` 없음 — Pipeline은 **노드 전용** |
| 모델을 `set_node(grp='xgb')`로 선언 | 모델은 **`Trial`** — Pipeline 밖, `exp()`에 직접 전달 |
| `p.build()` | `project.build_pipeline(p)` → 버전 부여 후 `e.set_pipeline(...)` |
| `Experimenter(df, sp=, path=)` | `project.experimenter(name, df, ...)` / `project.load_experimenter(name, df)` |
| `e.set_collector(...)` / `e.get_collector(...)` | `project.collectors()` 레지스트리 (프로젝트 전역, 등록 즉시 영속화) |
| `e.exp(nodes='xgb1')` | `e.exp([(trial, outer, inner), ...], project.trials, collectors=...)` |
| `e.exp(finalize=True)` | `finalize` 개념 없음 |
| 수집 실패는 `collector.warnings` (메모리) | `collectors.hist` — **`CollectHist`** (fold 단위 이력) |

grp 상속이 모델에 적용되지 않으므로, 그 자리는 아래 `MODELS` dict + `trial()` 헬퍼가 대신한다.

In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import mllabs

data_path = Path('data')

In [2]:
from mllabs.processor import PolarsLoader
ploader = PolarsLoader(predefined_types={'id': pl.Int64, 'obj_id': pl.Float64}, infer_schema_length=10000)
ploader.fit([data_path / 'train.csv', data_path / 'test.csv', data_path / 'star_classification.csv'])
df_train = ploader.transform([data_path / 'train.csv'])
df_test = ploader.transform([data_path / 'test.csv']).with_columns(
   pl.lit('').alias('class')
)

In [3]:
from itertools import combinations
from mllabs.processor import ExprProcessor
expr_dict1 = {
    'u': pl.when(pl.col('u') < 10).then(
        pl.when(pl.col("class") == "STAR").then(pl.col("u")).otherwise(None).mean()
    ).otherwise(pl.col('u')),
    'alpha90': pl.col('alpha') + 90,
    'spectral_type_galaxy_population': (pl.col('spectral_type').cast(pl.String) + '_' + pl.col('galaxy_population').cast(pl.String)).cast(pl.Categorical)
}
X_mags = ['u', 'g', 'r', 'i', 'z']
expr_dict2 = {
    'mag_mean': pl.mean_horizontal(*X_mags),
    'mag_std': pl.concat_list(X_mags).list.std(),
    'mag_min': pl.min_horizontal(*X_mags),
    'mag_max': pl.max_horizontal(*X_mags),
    'mag_range': pl.max_horizontal(*X_mags) - pl.min_horizontal(*X_mags),
}
X_mags_stat = list(expr_dict2.keys())
expr_dict2 = {
    **expr_dict2,
    'mag_vmax': pl.struct(X_mags).map_elements(
        lambda x: max(x, key=x.get),
        return_dtype=pl.String),
    'redshift_log': (pl.col('redshift') + 1e-1).log(),
    'redshift_1e-4': (pl.col('redshift') == 0.0001).cast(pl.Int8),
    'spectral_type_ord': pl.col('spectral_type').replace({'M': 0, 'G/K': 1, 'A/F': 2, 'O/B': 3}).to_physical().cast(pl.Int8),
    'galaxy_population_i': pl.when(pl.col('galaxy_population') == 'Red_Sequence').then(1).otherwise(0).cast(pl.Int8),
}
X_diff = list()
for i, j in combinations(X_mags, 2):
    X_diff.append(f'{i}_{j}')
    expr_dict2[X_diff[-1]] = pl.col(i) - pl.col(j)

X_mags_log = list()
for i in X_mags:
    X_mags_log.append(f'{i}_log')
    expr_dict2[X_mags_log[-1]] = pl.col(i).log()

In [4]:
from sklearn.pipeline import make_pipeline
expr_p = make_pipeline(ExprProcessor(expr_dict1), ExprProcessor(expr_dict2))
df_train = expr_p.fit_transform(df_train)
df_test = expr_p.transform(df_test)

In [5]:
import pickle as pkl
if not os.path.exists('data/lof.pkl'):
    from sklearn.neighbors import LocalOutlierFactor
    from sklearn.cluster import KMeans
    df_lof = pl.concat([df_train[['alpha90', 'delta']], df_test[['alpha90', 'delta']]])
    lof = LocalOutlierFactor()
    lof.fit(df_lof)
    lof_ = lof.negative_outlier_factor_
    clu_kmeans = KMeans(3000)
    clu_kmeans.fit(df_lof)
    km3000 = clu_kmeans.labels_
    with open('data/lof.pkl', 'wb') as f:
        pkl.dump((lof_, km3000), f)
else:
    with open('data/lof.pkl', 'rb') as f:
        lof_, km3000 = pkl.load(f)

df_train = df_train.with_columns(
    pl.Series('lof', lof_[:len(df_train)]),
    pl.Series('km3000', km3000[:len(df_train)], dtype=pl.String).cast(pl.Categorical)
)
df_test = df_test.with_columns(
    pl.Series('lof', lof_[len(df_train):]),
    pl.Series('km3000', km3000[len(df_train):], dtype=pl.String).cast(pl.Categorical)
)
df_train.head()

id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class,alpha90,spectral_type_galaxy_population,mag_mean,mag_std,mag_min,mag_max,mag_range,mag_vmax,redshift_log,redshift_1e-4,spectral_type_ord,galaxy_population_i,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z,u_log,g_log,r_log,i_log,z_log,lof,km3000
i64,f32,f32,f32,f32,f32,f32,f32,f32,cat,cat,cat,f32,cat,f32,f32,f32,f32,f32,str,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,cat
0,147.734253,16.959272,25.472122,21.895559,20.357925,19.257113,18.621058,0.408982,"""M""","""Red_Sequence""","""GALAXY""",237.734253,"""M_Red_Sequence""",21.120754,2.731221,18.621058,25.472122,6.851065,"""u""",-0.675342,0,0,1,3.576563,5.114197,6.21501,6.851065,1.537634,2.638447,3.274502,1.100813,1.736868,0.636055,3.237585,3.086284,3.01347,2.957881,2.924293,-1.265492,"""298"""
1,127.988678,32.346718,20.778509,19.087063,17.587208,17.226067,16.786432,0.157976,"""M""","""Red_Sequence""","""GALAXY""",217.988678,"""M_Red_Sequence""",18.293055,1.636653,16.786432,20.778509,3.992077,"""u""",-1.35489,0,0,1,1.691446,3.191301,3.552443,3.992077,1.499855,1.860996,2.300631,0.361141,0.800776,0.439634,3.03392,2.949011,2.867172,2.846424,2.820571,-1.085606,"""745"""
2,179.792648,35.344845,21.035202,21.079128,21.171841,20.58263,20.557365,2.82377,"""O/B""","""Blue_Cloud""","""QSO""",269.792664,"""O/B_Blue_Cloud""",20.885233,0.292103,20.557365,21.171841,0.614475,"""r""",1.072874,0,3,0,-0.043926,-0.136639,0.452572,0.477837,-0.092712,0.496498,0.521763,0.589211,0.614475,0.025265,3.046198,3.048284,3.052672,3.024448,3.02322,-1.186385,"""2279"""
3,225.818298,48.56942,23.305056,21.050735,19.017754,18.365658,17.914951,0.536099,"""M""","""Red_Sequence""","""GALAXY""",315.818298,"""M_Red_Sequence""",19.93083,2.235331,17.914951,23.305056,5.390104,"""u""",-0.452402,0,0,1,2.25432,4.287302,4.939398,5.390104,2.032982,2.685078,3.135784,0.652096,1.102802,0.450706,3.148671,3.046936,2.945373,2.910483,2.885636,-0.991541,"""560"""
4,141.836136,19.342852,21.703157,19.47168,18.234449,17.899446,17.616184,0.555761,"""M""","""Red_Sequence""","""GALAXY""",231.836136,"""M_Red_Sequence""",18.984983,1.676354,17.616184,21.703157,4.086973,"""u""",-0.421958,0,0,1,2.231478,3.468708,3.803711,4.086973,1.23723,1.572233,1.855495,0.335003,0.618265,0.283262,3.077458,2.968961,2.903313,2.88477,2.868818,-1.048082,"""2937"""


In [6]:
y2 = 'class'
class_weight = df_train[y2].to_pandas().value_counts().pipe(
    lambda x: x / x.min()
).to_dict()
class_weight

{'GALAXY': 4.56312557419854, 'QSO': 1.4160703060780426, 'STAR': 1.0}

In [7]:
y = 'class_i'
y_repl = {'STAR': 0, 'QSO': 1, 'GALAXY': 2}
df_train = df_train.with_columns(
    **{
        y: pl.col(y2).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y2).cast(pl.String).replace(class_weight).cast(pl.Float32)
    }
)

In [8]:
X_loc = ['alpha', 'delta']
X_num = ['redshift', 'redshift_log', 'lof', 'alpha90']
X_bin = ['redshift_1e-4', 'galaxy_population_i']
X_nom = ['galaxy_population', 'spectral_type', 'spectral_type_galaxy_population']
X_base = X_loc[1:] + X_num + X_bin + X_nom[1:] + X_diff + X_mags_stat + X_mags_log

X_nom2 = ['km3000']
X_ohe = ['spectral_type', 'spectral_type_galaxy_population']
X_std = ['redshift_log', 'lof'] + X_mags + X_mags_stat + X_mags_log + X_diff

X_num = X_loc + X_num + X_mags + X_mags_stat + X_mags_log + X_diff
X_nom = X_nom + X_nom2

## Project 구성

`Project`가 디렉토리 레이아웃과 프로젝트 전역인 것들(pipelines / collectors / TrialStore / cache)을 소유한다.
run 하나(Experimenter)에 관한 것 — splitter, 채택한 Pipeline, 노드 아티팩트 — 은 전부 `exp/phase2/` 안에 있어서
Project 없이도 열 수 있다.

In [9]:
# 처음부터 다시 돌릴 때만
# !rm -rf exp

In [10]:
from mllabs import Project, Trial
from mllabs import ProgressSessionLogger, TqdmProgressSession
from sklearn.model_selection import StratifiedShuffleSplit
from IPython.display import display, Markdown

logger = ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)

project = Project('exp')
p = project.pipeline_builder('phase2_pipeline')

In [11]:
p.set_datasource({
    **{i: 'numerical' for i in X_num},
    **{i: 'binary' for i in X_bin},
    **{i: 'nominal' for i in X_nom + [y, y2]},
}, targets=[y, y2])

'skip'

### Pipeline — 전처리 노드만

`role`이 없어진 뒤로 Pipeline에 담기는 건 전처리 노드뿐이다. 예전에 `role='head'`로 선언하던 모델 그룹
(`clf`/`xgb`/`lgb`/...)은 여기 들어가지 않는다.

### 모델 — Trial

`Trial`은 Pipeline 밖이라 **grp 상속이 없다.** 예전에 `set_grp('xgb', parent='clf', ...)`가 해주던
공통 파라미터 상속은 아래 `MODELS` dict와 `trial()` 헬퍼가 대신한다 — 어차피 파이썬 dict 병합이라
상속 규칙을 따로 외울 필요가 없어졌다.

`edges_for()`는 라운드마다 namespace 세그먼트(`tgt_delta:(*)` 등)를 덧붙이는 부분을 한곳에 모은 것이다.

In [12]:
def dsl_set(cols):
    return '{' + ', '.join(cols) + '}'


y_edges = {'y': dsl_set([y])}

p.set_grp('pre', method='transform')
p.set_grp('pre_ft', method='fit_transform', edges=y_edges)
p.set_grp('pre_ft2', method='fit_transform', edges={'y': dsl_set([y2])})

p.set_node('std', grp='pre', processor='sklearn.preprocessing.StandardScaler', edges={'X': dsl_set(X_std)})
p.set_node('ohe', grp='pre', processor='sklearn.preprocessing.OneHotEncoder', edges={'X': dsl_set(X_ohe)}, params={'sparse_output': False})
p.set_node('coov', grp='pre', processor='mllabs.processor.CatOOVFilter', edges={'X': dsl_set(X_nom)})
p.set_node('tgt_km3000', grp='pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges={'X': dsl_set(X_nom2)}, params={'target_type': 'multiclass'})

{'result': 'skip',
 'affected_nodes': [],
 'old_obj': <mllabs._pipeline._PipelineNode at 0x7f653db2a900>,
 'obj': <mllabs._pipeline._PipelineNode at 0x7f653db2a900>}

In [13]:
MODELS = {
    'xgb': dict(
        processor='xgboost.XGBClassifier',
        adapter={'__ref__': 'mllabs.adapter.XGBoostAdapter', '__params__': {'eval_mode': 'both'}},
        params={'random_state': 123, 'n_estimators': 10000, 'enable_categorical': True,
                'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'},
    ),
    'lgb': dict(
        processor='lightgbm.LGBMClassifier',
        adapter={'__ref__': 'mllabs.adapter.LightGBMAdapter', '__params__': {'eval_mode': 'both'}},
        params={'random_state': 123, 'n_estimators': 10000, 'verbose': -1, 'gpu': None,
                'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},
                'eval_metric': 'multi_logloss'},
    ),
    'cb': dict(
        processor='catboost.CatBoostClassifier',
        adapter={'__ref__': 'mllabs.adapter.CatBoostAdapter', '__params__': {'eval_mode': 'valid'}},
        params={'random_state': 123, 'n_estimators': 10000, 'early_stopping_rounds': 50,
                'eval_metric': 'AUC', 'verbose': 0,
                'cat_features': {'__ref__': 'mllabs.ColSelector',
                                 '__params__': {'dsl_string': '*@categorical'}}},
    ),
    'nn': dict(
        processor='mllabs.nn.NNClassifier',
        params={'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10, 'epochs': 200},
    ),
    'lr': dict(processor='sklearn.linear_model.LogisticRegression', params={}),
    'dt': dict(processor='sklearn.tree.DecisionTreeClassifier', params={'random_state': 123}),
}


def trial(name, model, X, **params):
    m = MODELS[model]
    return Trial(name, m['processor'], {'X': X, **y_edges}, method='predict',
                 adapter=m.get('adapter'), params={**m['params'], **params})


def edges_for(*extra, xgb_cols=None):
    ns = ''.join(f' + {n}:(*)' for n in extra)
    linear = 'std:(*) + ohe:(*@ohe_drop_first)'
    return {
        'xgb': dsl_set(xgb_cols if xgb_cols is not None else X_num + X_bin) + ' + coov:(*)' + ns,
        'lgb': dsl_set(X_base) + ns,
        'cb': dsl_set(X_base) + ns,
        'nn': linear + ns,
        'lr': linear + ns,
    }


def round_trials(idx, *extra, **kw):
    E = edges_for(*extra, **kw)
    return [trial(f'{m}{idx}', m, E[m]) for m in ('xgb', 'lgb', 'cb', 'nn', 'lr')]

In [14]:
collectors = project.collectors()


def conn(**kw):
    return {'__ref__': 'mllabs.Connector', '__params__': kw}


MA = 'mllabs.collector.ModelAttrCollector'
collectors.set_collector('xgb_evals_results', MA, conn(processor='xgboost.XGBClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector('lgb_evals_results', MA, conn(processor='lightgbm.LGBMClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector('cb_evals_results', MA, conn(processor='catboost.CatBoostClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector('nn_evals', MA, conn(processor='mllabs.nn.NNClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector(
    'bAcc', 'mllabs.collector.MetricCollector', conn(edges=y_edges),
    params={'output_var': '-1:',
            'metric_func': {'__callable__': 'sklearn.metrics.balanced_accuracy_score'},
            'include_train': True})
collectors.set_collector('lgb_feature_importance', MA, conn(processor='lightgbm.LGBMClassifier', edges=y_edges), params={'result_key': 'feature_importances'})
collectors.set_collector('xgb_feature_importance_gain', MA, conn(processor='xgboost.XGBClassifier', edges=y_edges), params={'result_key': 'feature_importances', 'params': {'importance_type': 'gain'}})
collectors.set_collector('xgb_feature_importance_cover', MA, conn(processor='xgboost.XGBClassifier', edges=y_edges), params={'result_key': 'feature_importances', 'params': {'importance_type': 'cover'}})
collectors.set_collector('cb_feature_importance', MA, conn(processor='catboost.CatBoostClassifier', edges=y_edges), params={'result_key': 'feature_importances_pvc'})
collectors.set_collector('cb_interaction', MA, conn(processor='catboost.CatBoostClassifier', edges=y_edges), params={'result_key': 'feature_importances_interaction'})
collectors.set_collector('lr_coef', MA, conn(processor='sklearn.linear_model.LogisticRegression', edges=y_edges), params={'result_key': 'coef'})

collectors.names()

2026-08-04 04:38:52.765781: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


['xgb_evals_results',
 'lgb_evals_results',
 'cb_evals_results',
 'nn_evals',
 'bAcc',
 'lgb_feature_importance',
 'xgb_feature_importance_gain',
 'xgb_feature_importance_cover',
 'cb_feature_importance',
 'cb_interaction',
 'lr_coef']

`set_collector`는 **등록 즉시 영속화**된다(`collectors/collectors.db` + `__params/{name}.pkl`).
`Collectors.save()`는 없어졌고, 다음 세션에서는 `project.collectors()` 호출 자체가 복원이다.

`Connector(role='head')`의 `role`도 없어졌다 — Collector는 애초에 Trial job에만 붙고 노드 job엔 안 붙어서
걸러야 할 대상이 없었다.

In [15]:
def adopt():
    """현재 builder 상태를 새 버전으로 빌드해 이 run에 채택시킨다.
    바뀐 노드만 stale 처리되어 아티팩트가 지워진다 (Pipeline.diff_from)."""
    e.set_pipeline(project.build_pipeline(p), 'phase2_pipeline')
    return e.pipeline_version


if 'phase2' in project.list_experimenters():
    e = project.load_experimenter('phase2', df_train)
else:
    e = project.experimenter(
        'phase2', df_train,
        sp=StratifiedShuffleSplit(n_splits=1, random_state=123, train_size=0.8),
        sp_v=StratifiedShuffleSplit(n_splits=1, random_state=123, train_size=0.9),
        splitter_params={'y': y},
        pipeline_name='phase2_pipeline',
        pipeline_version=project.build_pipeline(p).version,
    )
e.pipeline_version

7

In [16]:
def folds(trials):
    return [(t, o, i) for t in trials
            for o in range(e.get_n_splits())
            for i in range(e.get_n_splits_inner())]


def run(trials, **kw):
    """collectors를 레지스트리째 넘기면 수집 이력도 collectors.hist에 자동으로 남는다."""
    kw = {'n_jobs': 2, 'gpu_id_list': [0], 'logger': logger, **kw}
    e.exp(folds(trials), project.trials, collectors=collectors, **kw)

In [17]:
display(Markdown(p.desc_node('tgt_km3000')))
display(Markdown(p.desc_pipeline()))

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_tgt_km3000["pre_ft2/tgt_km3000"]
        tgt_km3000_dummy[ ]
        style tgt_km3000_dummy fill:none,stroke:none
    end
    style node_tgt_km3000 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    DataSource -->|X,y| node_tgt_km3000
```

**Path from DataSource to 'pre_ft2/tgt_km3000' (1 path(s) found)**

### Edges

| Key | Node | Var |
|-----|------|-----|
| X | Data Source | `{km3000}` |
| y | Data Source | `{class}` |

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_pre["pre"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_coov["coov"]
        style node_coov fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_delta_100["delta_100"]
        style node_delta_100 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_delta_500["delta_500"]
        style node_delta_500 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_alpha_100["alpha_100"]
        style node_alpha_100 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_alpha_500["alpha_500"]
        style node_alpha_500 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_cat0["cat0"]
        style node_cat0 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_cat1["cat1"]
        style node_cat1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe_coov["ohe_coov"]
        style node_ohe_coov fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre_ft2["pre_ft2"]
        node_tgt_km3000["tgt_km3000"]
        style node_tgt_km3000 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_tgt_delta["tgt_delta"]
        style node_tgt_delta fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_tgt_alpha["tgt_alpha"]
        style node_tgt_alpha fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_lda_mags["lda_mags"]
        style node_lda_mags fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre_ft2 fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp___datasource__["__datasource__"]
    end
    style grp___datasource__ fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre_ft["pre_ft"]
    end
    style grp_pre_ft fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    DataSource --> grp_pre
    DataSource --> grp_pre_ft2
    grp_pre --> grp_pre_ft2
```

In [18]:
with e.os_log():
    e.build()

No stage nodes to build


## Round 1 — 기본 피처

In [19]:
round1 = round_trials(1, xgb_cols=X_num)
with e.os_log():
    run([t for t in round1 if t.name == 'xgb1'])

No trials to run


In [20]:
with e.os_log():
    run(round1)

No trials to run


## Round 2 — `tgt_km3000` 투입

노드는 그대로라 파이프라인을 다시 빌드할 필요가 없다. Trial만 새로 만들어 돌린다.

In [21]:
round2 = round_trials(2, 'tgt_km3000')
with e.os_log():
    run(round2)

No trials to run


In [22]:
collectors.get_collector('lgb_feature_importance').get_attrs_agg('lgb2').sort_values(ascending=False).iloc[:10]

delta                        806.0
redshift                     722.0
alpha90                      650.0
redshift_log                 591.0
tgt_km3000__km3000_GALAXY    449.0
tgt_km3000__km3000_STAR      415.0
g_z                          415.0
g_log                        402.0
u_g                          365.0
r_z                          317.0
dtype: float64

In [23]:
collectors.get_collector('xgb_feature_importance_gain').get_attrs_agg('xgb2').sort_values(ascending=False).iloc[:10]

g_z                          619.345337
redshift                     375.374542
redshift_1e-4                182.485519
u_i                          146.965561
r_log                        144.687973
mag_std                      137.623642
tgt_km3000__km3000_GALAXY     88.602722
tgt_km3000__km3000_STAR       74.811714
g_i                           69.876015
z_log                         68.417618
dtype: float64

In [24]:
collectors.get_collector('lr_coef').get_attrs_agg('lr2').sort_values(ascending=False).iloc[:10]

2  std__u_log                 6.939461
1  std__r_log                 6.507367
   std__z_log                 6.212572
   std__i_log                 5.927530
   std__g_log                 4.480608
0  std__z_log                 3.807741
2  std__z                     2.645722
   std__i                     2.565824
0  tgt_km3000__km3000_STAR    2.420721
2  std__r                     2.357667
dtype: float64

In [25]:
collectors.get_collector('cb_interaction').get_attrs_agg('cb2').sort_values(ascending=False).iloc[:20]

feat1     feat2                    
redshift  g_z                          1.741178
delta     alpha90                      1.691145
redshift  tgt_km3000__km3000_GALAXY    1.372492
          redshift_1e-4                1.280813
delta     redshift                     1.278790
redshift  r_z                          1.212415
          tgt_km3000__km3000_STAR      1.181098
          g_log                        1.163935
          i_log                        1.044857
          alpha90                      0.991820
          mag_min                      0.973341
          r_log                        0.934054
          z_log                        0.908830
delta     tgt_km3000__km3000_GALAXY    0.845167
redshift  g_i                          0.831704
          u_i                          0.827762
          u_g                          0.753792
delta     tgt_km3000__km3000_STAR      0.711200
redshift  mag_max                      0.703501
          mag_std                      0.695335
dtyp

## Round 3 — `delta` 타겟 인코딩

여기서부터는 노드가 추가되므로 `adopt()`(= `build_pipeline` + `set_pipeline`)를 거쳐야 한다.
Pipeline은 불변이라 builder 수정이 진행 중인 run에 새어 들어가지 않는다.

In [26]:
with e.os_log():
    p.set_node('delta_100', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['delta'])}, params={'n_bins': 100, 'encode': 'ordinal'})
    p.set_node('delta_500', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['delta'])}, params={'n_bins': 500, 'encode': 'ordinal'})
    p.set_node('tgt_delta', grp='pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges={'X': 'delta_100:(*) + delta_500:(*)'}, params={'target_type': 'multiclass'})
    adopt()
    e.build()

No stage nodes to build


In [27]:
round3 = round_trials(3, 'tgt_km3000', 'tgt_delta')
with e.os_log():
    run(round3)

No trials to run


## Round 4 — `alpha90` 타겟 인코딩

In [28]:
with e.os_log():
    p.set_node('alpha_100', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['alpha90'])}, params={'n_bins': 100, 'encode': 'ordinal'})
    p.set_node('alpha_500', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['alpha90'])}, params={'n_bins': 500, 'encode': 'ordinal'})
    p.set_node('tgt_alpha', grp='pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges={'X': 'alpha_100:(*) + alpha_500:(*)'}, params={'target_type': 'multiclass'})
    adopt()
    e.build()

No stage nodes to build


In [29]:
round4 = round_trials(4, 'tgt_km3000', 'tgt_delta', 'tgt_alpha')
with e.os_log():
    run(round4)

No trials to run


In [30]:
collectors.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending=False).iloc[:15]

,test,train,valid
cb4,0.958274,0.964901,0.957964
cb5,0.957950,0.962039,0.956905
cb3,0.957855,0.963015,0.957307
lgb5,0.957563,0.961974,0.956872
lgb4,0.957270,0.961500,0.956604
lgb3,0.956999,0.961119,0.956334
cb2,0.956966,0.963454,0.956432
lgb2,0.956792,0.961654,0.956652
xgb5,0.955728,0.973664,0.953765
xgb4,0.954848,0.975980,0.954221


## Round 5 — LDA + 범주형 좌표 bin

In [31]:
with e.os_log():
    p.set_node('cat0', grp='pre', processor='mllabs.processor.CatConverter', edges={'X': 'alpha_500:(*) + delta_500:(*)'})
    p.set_node('cat1', grp='pre', processor='mllabs.processor.CatConverter', edges={'X': 'alpha_100:(*) + delta_100:(*)'})
    p.set_node('lda_mags', grp='pre_ft2', processor='sklearn.discriminant_analysis.LinearDiscriminantAnalysis', edges={'X': dsl_set(X_mags)})
    adopt()
    e.build()

No stage nodes to build


In [32]:
EXTRA5 = ('tgt_km3000', 'tgt_delta', 'tgt_alpha', 'lda_mags')
round5 = round_trials(5, *EXTRA5)

nn_base = edges_for(*EXTRA5)['nn']
cat_cols = {'__ref__': 'mllabs.ColSelector', '__params__': {'dsl_string': '*@categorical'}}
round5 += [
    trial('nn6', 'nn', nn_base + ' + ' + dsl_set(X_nom2) + ' + cat0:(*)', cat_cols=cat_cols),
    trial('nn7', 'nn', nn_base + ' + ' + dsl_set(X_nom2) + ' + cat1:(*)', cat_cols=cat_cols),
    trial('nn8', 'nn', nn_base + ' + ' + dsl_set(X_nom2), cat_cols=cat_cols),
    trial('nn9', 'nn', nn_base + ' + cat1:(*)', cat_cols=cat_cols),
]

with e.os_log():
    run(round5)

No trials to run


In [33]:
collectors.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending=False)

,test,train,valid
cb4,0.958274,0.964901,0.957964
cb5,0.957950,0.962039,0.956905
cb3,0.957855,0.963015,0.957307
lgb5,0.957563,0.961974,0.956872
lgb4,0.957270,0.961500,0.956604
lgb3,0.956999,0.961119,0.956334
cb2,0.956966,0.963454,0.956432
lgb2,0.956792,0.961654,0.956652
xgb5,0.955728,0.973664,0.953765
xgb4,0.954848,0.975980,0.954221


## dtype 기반 selector (`@numeric` / `@categorical` / `@binary` / `@float` / `@int` / `@string`)

edges DSL에서 `@numeric`/`@categorical` 등은 실제 컬럼의 dtype을 보고 선택하는 selector다 (processor 불필요).
DataSource 최상위(`*@numeric`)에 바로 걸면 `id`/`class_i`(target)/`sample_weight`처럼 스키마에 없는 raw 컬럼까지
딸려 들어올 수 있어 위험하므로, 이미 확정된 **노드 출력 namespace 안에서만** 사용한다.

In [34]:
with e.os_log():
    p.set_node('ohe_coov', grp='pre', processor='sklearn.preprocessing.OneHotEncoder', edges={'X': 'coov:(*@categorical)'}, 
               params={'sparse_output': False, 'handle_unknown': 'ignore'})
    adopt()
    e.build()

No stage nodes to build


In [44]:
dt1 = trial('dt1', 'dt', 'std:(*@numeric) + ohe_coov:(* - ^.*__km3000.*$) + tgt_km3000:(*)')
run([dt1])
collectors.get_collector('bAcc').get_metrics_agg('dt1')[0]

Experimenting 1 job(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Exp complete: 1 job(s)


,test,train,valid
dt1,0.927173,1.0,0.928124


## 수집 이력 — `CollectHist`

`collectors.hist`는 `(collector, experimenter, node, outer_idx, inner_idx)`마다 한 행을 남긴다.

- `status`: `'collected'` / `'empty'`(예외 없이 `None` 반환 — 보통 `output_var` 설정 실수) / `'error'`
- `info`: 에러일 때 `{phase, type, message, traceback}`. `phase`는 `'output'`/`'ext'`/`'collect'`/`'push'`
- `elapsed`: `collect()` 호출 시간

예전엔 수집 실패가 `collector.warnings`(메모리)에만 쌓였고, 멀티워커에선 워커 사본에 쌓였다가 그대로
버려졌다. 지금은 항상 부모 프로세스가 기록한다.

In [57]:
hist = collectors.hist
pd.DataFrame(hist.get_hist()).groupby(['collector_name', 'status']).size().unstack(fill_value=0)

status,collected,error
collector_name,,
bAcc,30,0
cb_evals_results,3,2
cb_feature_importance,5,0
cb_interaction,5,0
lgb_evals_results,5,0
lgb_feature_importance,5,0
lr_coef,5,0
nn_evals,9,0
xgb_evals_results,5,0


In [47]:
for row in hist.get_hist(status='error'):
    print(row['collector_name'], row['node_name'], (row['outer_idx'], row['inner_idx']),
          row['info']['phase'], row['info']['type'], row['info']['message'])

cb_evals_results cb2 (0, 0) collect ValueError All arrays must be of the same length
cb_evals_results cb4 (0, 0) collect ValueError All arrays must be of the same length


In [48]:
pd.DataFrame(hist.get_hist()).groupby('collector_name')['elapsed'].agg(['sum', 'mean', 'max']).sort_values('sum', ascending=False)

,sum,mean,max
collector_name,,,
bAcc,1.246023,0.041534,0.056361
cb_feature_importance,0.081687,0.016337,0.054354
cb_interaction,0.057519,0.011504,0.013062
nn_evals,0.037641,0.004182,0.007253
cb_evals_results,0.021640,0.004328,0.008124
xgb_evals_results,0.016474,0.003295,0.003421
lgb_evals_results,0.014436,0.002887,0.004020
lr_coef,0.006040,0.001208,0.001623
xgb_feature_importance_gain,0.004897,0.000979,0.002067


### 나중에 붙인 Collector가 아무것도 못 보는 경우

수집은 Trial job이 실행될 때의 부수효과다. `TrialStore.experiment_hist`에 이미 `'built'`로 기록된 fold는
`exp()`가 스킵하므로, 실험이 끝난 뒤에 Collector를 새로 붙이고 `exp()`를 다시 불러봐야 아무것도 수집되지
않는다. `CollectHist`를 조회하면 그 사실이 드러난다.

다시 수집하려면 그 Trial의 fold 이력을 명시적으로 지워야 한다 — `reset_nodes()`는 아티팩트만 지우고
스킵 판정에 쓰이는 이력은 건드리지 않는다.

In [49]:
collectors.set_collector('cb_evals_results', MA, conn(processor='catboost.CatBoostClassifier'),
                         params={'result_key': 'evals_result'}, exist='replace')

cb_trials = [t for t in round1 + round2 + round3 + round4 + round5 if t.name.startswith('cb')]
missing = [t for t in cb_trials
           if not hist.get_hist(collector_name='cb_evals_results', node_name=t.name)]
[t.name for t in missing]

[]

In [50]:
for t in missing:
    project.trials.remove_hist(trial_name=t.name, experimenter=e.name)
e.reset_nodes([t.name for t in missing])

with e.os_log():
    run(missing)

No trials to run


In [60]:
collectors.get_collector('cb_evals_results').get_attrs('cb2')

{}

## 상태 점검

Trial 실행 이력은 `TrialStore`(프로젝트 전역)에, 노드 이력은 이 run의 `NodeStore`에 있다.

In [52]:
e.show_error_nodes(trial_store=project.trials)
display(Markdown(e.get_node_info()))

# Experiment Pipeline Summary

- **DataSource**

## std
- **Processor**: sklearn.preprocessing.StandardScaler
- **Method**: transform
- **Edges**: X: {redshift_log, lof, u, g, r, i, z, mag_mean, mag_std, mag_min, mag_max, mag_range, u_log, g_log, r_log, i_log, z_log, u_g, u_r, u_i, u_z, g_r, g_i, g_z, r_i, r_z, i_z}

## ohe
- **Processor**: sklearn.preprocessing.OneHotEncoder
- **Method**: transform
- **Edges**: X: {spectral_type, spectral_type_galaxy_population}

## coov
- **Processor**: mllabs.processor.CatOOVFilter
- **Method**: transform
- **Edges**: X: {galaxy_population, spectral_type, spectral_type_galaxy_population, km3000}
- **Descendants**: ['ohe_coov']

## tgt_km3000
- **Processor**: sklearn.preprocessing.TargetEncoder
- **Method**: fit_transform
- **Edges**: y: {class}, X: {km3000}

## delta_100
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {delta}
- **Descendants**: ['cat1', 'tgt_delta']

## delta_500
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {delta}
- **Descendants**: ['cat0', 'tgt_delta']

## tgt_delta
- **Processor**: sklearn.preprocessing.TargetEncoder
- **Method**: fit_transform
- **Edges**: y: {class}, X: delta_100:(*) + delta_500:(*)

## alpha_100
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {alpha90}
- **Descendants**: ['cat1', 'tgt_alpha']

## alpha_500
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {alpha90}
- **Descendants**: ['cat0', 'tgt_alpha']

## tgt_alpha
- **Processor**: sklearn.preprocessing.TargetEncoder
- **Method**: fit_transform
- **Edges**: y: {class}, X: alpha_100:(*) + alpha_500:(*)

## cat0
- **Processor**: mllabs.processor.CatConverter
- **Method**: transform
- **Edges**: X: alpha_500:(*) + delta_500:(*)

## cat1
- **Processor**: mllabs.processor.CatConverter
- **Method**: transform
- **Edges**: X: alpha_100:(*) + delta_100:(*)

## lda_mags
- **Processor**: sklearn.discriminant_analysis.LinearDiscriminantAnalysis
- **Method**: fit_transform
- **Edges**: y: {class}, X: {u, g, r, i, z}

## ohe_coov
- **Processor**: sklearn.preprocessing.OneHotEncoder
- **Method**: transform
- **Edges**: X: coov:(*@categorical)


In [53]:
pd.DataFrame(project.trials.get_hist(experimenter=e.name))[
    ['trial_name', 'outer_idx', 'inner_idx', 'pipeline_version', 'status']
]

,trial_name,outer_idx,inner_idx,pipeline_version,status
0,cb1,0,0,1,built
1,cb2,0,0,1,built
2,cb3,0,0,2,built
3,cb4,0,0,3,built
4,cb5,0,0,4,built
5,dt1,0,0,11,built
6,lgb1,0,0,1,built
7,lgb2,0,0,1,built
8,lgb3,0,0,2,built
9,lgb4,0,0,3,built
